<img src="">

<div style="display:fill;
           background-color:#e1d9ce;
           letter-spacing:0.5px;border-bottom: 2px solid black;">
<img src="https://images.unsplash.com/photo-1501167786227-4cba60f6d58f?q=80&h=500&w=2000&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D">
    
<H1 style="padding: 10px; color:black; font-weight:600;font-family: 'Garamond', 'Lucida Sans', sans-serif; text-align: center; font-size: 42px;">Binary Classification with a Bank Churn Dataset</H1>
</div>


In [ ]:
import numpy as np 
import pandas as pd 
import warnings
warnings.filterwarnings("ignore")
import os
pd.plotting.register_matplotlib_converters()
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_style("dark") # Theme for plots as Dark
sns.set_palette("viridis")
# sns.color_palette("flare")
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, cross_validate, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
import optuna
import imblearn
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from catboost import Pool, CatBoostClassifier, cv
from imblearn.over_sampling import KMeansSMOTE
from imblearn.over_sampling import SMOTE, ADASYN
from collections import Counter
from sklearn.cluster import KMeans

<div style="background-color: #e1d9ce; padding: 20px; border-radius: 20px; border: 2px solid black;">
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: black; font-weight: bold; font-size: 42px;">
    Table of Contents
    </h1>
</div>

<a href="#1" style="font-family: 'Lucida Sans', 'Lucida Sans', sans-serif; text-align: left; color: #323232;font-size: 22px;"> 1. Dataset Overview </a><br>
<a href="#2" style="font-family: 'Lucida Sans', 'Lucida Sans', sans-serif; text-align: left; color: #323232;font-size: 22px;"> 2. Data Processing </a> <br>
<a href="#3" style="font-family: 'Lucida Sans', 'Lucida Sans', sans-serif; text-align: left; color: #323232;font-size: 22px;"> 3. Exploratory Data Analysis & Visualization </a> <br>
<a href="#4" style="font-family: 'Lucida Sans', 'Lucida Sans', sans-serif; text-align: left; color: #323232;font-size: 22px;"> 4. Training Models </a><br>
<a href="#4.1" style="font-family: 'Lucida Sans', 'Lucida Sans', sans-serif; text-align: left; color: #323232;font-size: 16px;padding-left: 25px;"> 4.1 Baseline Models  </a><br>
<a href="#4.2" style="font-family: 'Lucida Sans', 'Lucida Sans', sans-serif; text-align: left; color: #323232;font-size: 16px;padding-left: 25px;"> 4.2 Creating More Training Data from Test </a><br>
<a href="#4.3" style="font-family: 'Lucida Sans', 'Lucida Sans', sans-serif; text-align: left; color: #323232;font-size: 16px;padding-left: 25px;"> 4.3 Optuna-Tuned LGBM Model</a><br>
<a href="#4.4" style="font-family: 'Lucida Sans', 'Lucida Sans', sans-serif; text-align: left; color: #323232;font-size: 16px;padding-left: 25px;"> 4.4 Optuna-Tuned XGB Model </a><br>
<a href="#4.5" style="font-family: 'Lucida Sans', 'Lucida Sans', sans-serif; text-align: left; color: #323232;font-size: 16px;padding-left: 25px;"> 4.5 OOF LGBM+CatBoost+XGB Test Preds </a><br>
<a href="#7" style="font-family: 'Lucida Sans', 'Lucida Sans', sans-serif; text-align: left; color: #323232;font-size: 22px;"> 5. Creating 'submission.csv' </a><br>


<div id="1" style="background-color: #e1d9ce; padding: 20px; border-radius: 20px; border: 2px solid black;">
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #000000; font-weight: bold; font-size: 42px;">
    Dataset Overview
    </h1>
</div>

In [ ]:
train_data = pd.read_csv("/kaggle/input/playground-series-s4e1/train.csv",index_col="id")
test_data = pd.read_csv("/kaggle/input/playground-series-s4e1/test.csv",index_col="id")
orig_data = pd.read_csv("/kaggle/input/bank-customer-churn-prediction/Churn_Modelling.csv",index_col = "RowNumber")
orig_data.dropna(inplace=True)

# train_data.drop(["Surname"],inplace=True,axis=1)
# test_data.drop(["Surname"],inplace=True,axis=1)
# orig_data.drop(["Surname"],inplace=True,axis=1)

train_data = pd.concat([train_data,orig_data])
train_data.reset_index(drop=True,inplace=True)

In [ ]:
train_data.head()

In [ ]:
test_data.head()

<div id="2" style="background-color: #e1d9ce; padding: 20px; border-radius: 20px; border: 2px solid black;">
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #000000; font-weight: bold; font-size: 42px;">
    Data Processing
    </h1>
</div>

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

label_enc = LabelEncoder()

train_data["Gender"] = label_enc.fit_transform(train_data[["Gender"]])
test_data["Gender"] = label_enc.transform(test_data[["Gender"]])
train_data["Geography"] = label_enc.fit_transform(train_data[["Geography"]])
test_data["Geography"] = label_enc.transform(test_data[["Geography"]])
train_data["Surname"] = label_enc.fit_transform(train_data[["Surname"]])
test_data["Surname"] = label_enc.transform(test_data[["Surname"]])

train_data.head()

In [ ]:
N_CLUSTERS = 7
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=0, n_init="auto")

clusters_train = kmeans.fit_predict(train_data.drop(["Exited"],axis=1))
clusters_test = kmeans.predict(test_data)

train_data["cluster"] = clusters_train
test_data["cluster"] = clusters_test

<div id="3" style="background-color: #e1d9ce; padding: 20px; border-radius: 20px; border: 2px solid black;">
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #000000; font-weight: bold; font-size: 42px;">
    Exploratory Data Analysis & Visualization 
    </h1>
</div>

In [ ]:
mask = np.triu(np.ones_like(train_data.corr()))
plt.figure(figsize=(20,12))
sns.heatmap(train_data.corr(), cmap="copper", annot=True, mask=mask,vmin=-1,vmax=1);

In [ ]:
sns.jointplot(data=train_data, x="CreditScore", y="Balance", hue = "Exited", palette = "copper");

In [ ]:
sns.jointplot(data=train_data, x="EstimatedSalary", y="Balance", hue = "Exited", palette = "copper");

In [ ]:
sns.jointplot(data=train_data, x="CreditScore", y="Age", hue = "Exited", palette = "copper");

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(15,5),gridspec_kw={'width_ratios': [1, 2.5]})

plt.subplot(1,2,1)
sns.countplot(data=train_data,x="Exited", palette = "copper");
plt.title("Distribution of Exited")

plt.subplot(1,2,2)
sns.histplot(data=train_data,x="Age",hue="Exited", bins=40, kde=True, palette = "copper");
plt.title("Distribution of Age")

fig.show();

In [ ]:
clusts = list(dict(sorted(Counter(clusters_test).items(), key=lambda item: item[0])).values())
clusts.extend(list(dict(sorted(Counter(clusters_train).items(), key=lambda item: item[0])).values()))

clusters = pd.DataFrame()
clusters["clusters_type"] = ["test"]*N_CLUSTERS + ["train"]*N_CLUSTERS
clusters["Cluster Label"] = list(range(0,N_CLUSTERS))*2
clusters["No of Items"] = clusts

In [ ]:
plt.figure(figsize=(15,6))
sns.barplot(clusters,x="Cluster Label",y="No of Items",hue="clusters_type",palette = "copper")
plt.title("KMeans Clusters of Train & Test Data");

In [ ]:
clusters["No of Exited"] = 0
clusters["No of Exited"][clusters["clusters_type"]=="train"] = list(dict(sorted(Counter(train_data[train_data["Exited"]==1].cluster).items(), key=lambda item: item[0])).values())
clusters["Ratio of Exited"] = clusters["No of Exited"]/clusters["No of Items"]

In [ ]:
sns.barplot(data = clusters.query("clusters_type == 'train'"),y="Ratio of Exited", x ="Cluster Label",palette="copper");

<div id="4" style="background-color: #e1d9ce; padding: 20px; border-radius: 20px; border: 2px solid black;">
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #000000; font-weight: bold; font-size: 42px;">
   Training Models
    </h1>
</div>

In [ ]:
seed = np.random.seed(6)

X = train_data.drop("Exited",axis=1)
y = train_data.Exited

X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.3)
print(len(X_train),len(X_val))

<div id="4.1" >
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #263A29; font-weight: bold; font-size: 36px;">
   4.1 Baseline Models
    </h1>
</div>
<hr>

In [ ]:
lgbmmodel = LGBMClassifier(random_state=seed,device="gpu",verbose=-1)
lgbmmodel.fit(X_train,y_train)
print("\n","-"*25,"Baseline LGBM","-"*25)
print("CV score of LGBM is ",cross_val_score(lgbmmodel,X,y,cv=4, scoring = 'roc_auc').mean())
print("ROC AUC over Val Data:",roc_auc_score(lgbmmodel.predict(X_val),y_val))

xgbmodel = XGBClassifier(random_state=seed,tree_method= 'gpu_hist')
xgbmodel.fit(X_train,y_train)
print("\n","-"*25,"Baseline XGB","-"*25)
print("CV score of XGB is ",cross_val_score(xgbmodel,X,y,cv=4, scoring = 'roc_auc').mean())
print("ROC AUC over Val Data:",roc_auc_score(xgbmodel.predict(X_val),y_val))

<div id="4.2" >
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #263A29; font-weight: bold; font-size: 36px;">
   4.2 Creating More Training Data from Test
    </h1>
</div>
<hr>

<p href="#1" style="font-family: 'Lucida Sans', 'Lucida Sans', sans-serif; text-align: left; color: #323232;font-size: 14px;"> 
This notebook uses works of the following notebooks along with the my code:
    <ul>
<li><a href = "https://www.kaggle.com/code/kdmitrie/pgs41-just-few-lines-of-autogluon">https://www.kaggle.com/code/kdmitrie/pgs41-just-few-lines-of-autogluon</a><br>
<li><a href = "https://www.kaggle.com/code/arunklenin/ps4e1-advanced-feature-engineering-ensemble">https://www.kaggle.com/code/arunklenin/ps4e1-advanced-feature-engineering-ensemble</a>
        </ul>
</p><br>

In [ ]:
public_work1 = pd.read_csv("/kaggle/input/pgs41-just-few-lines-of-autogluon/simple_ag.csv")
public_work2 = pd.read_csv("/kaggle/input/ps4e1-advanced-feature-engineering-ensemble/submission.csv")

for i in range(5):
    print(f"> Generating More Data")
    print(f"  Inital Train Size:{len(train_data)} --> ",end="")
    
    X = train_data.drop("Exited",axis=1)
    y = train_data.Exited
    
    lgbmmodel.fit(X,y)
    xgbmodel.fit(X,y)
    
    more_train = test_data.copy()
    more_train["Exited"] = (lgbmmodel.predict_proba(test_data)[:,1]+xgbmodel.predict_proba(test_data)[:,1])/2
    more_train["Exited"] = (2*more_train["Exited"].to_numpy()+public_work1["Exited"].to_numpy()+public_work2["Exited"].to_numpy())/4
    more_train = more_train.query("Exited>0.95 | Exited<0.05")
    more_train["Exited"] = round(more_train["Exited"])
    more_train["Exited"] = more_train["Exited"].astype("int64")

    train_data = pd.concat([train_data,more_train])
    train_data.drop_duplicates(inplace=True)    
    train_data.reset_index(inplace=True,drop=True)
    train_data = train_data.sample(frac=1)
    
    print(f" Final Train Size:{len(train_data)}\n")

In [ ]:
X = train_data.drop("Exited",axis=1)
y = train_data.Exited

X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.3)
print(len(X_train),len(X_val))

In [ ]:
lgbmmodel = LGBMClassifier(random_state=seed,device="gpu",verbose=-1)
lgbmmodel.fit(X_train,y_train)
print("\n","-"*25,"New Baseline Scores for LGBM","-"*25)
print("CV score of LGBM is ",cross_val_score(lgbmmodel,X,y,cv=4, scoring = 'roc_auc').mean())
print("ROC AUC over Val Data:",roc_auc_score(lgbmmodel.predict(X_val),y_val))

xgbmodel = XGBClassifier(random_state=seed,tree_method= 'gpu_hist')
xgbmodel.fit(X_train,y_train)
print("\n","-"*25,"New Baseline Scores for XGB","-"*25)
print("CV score of XGB is ",cross_val_score(xgbmodel,X,y,cv=4, scoring = 'roc_auc').mean())
print("ROC AUC over Val Data:",roc_auc_score(xgbmodel.predict(X_val),y_val))

<div id="4.3" >
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #263A29; font-weight: bold; font-size: 36px;">
   4.3 Optuna-Tuned LGBM Classifier
    </h1>
</div>
<hr>

In [ ]:
# def objective(trial):
#     params = {
#         'n_estimators' : trial.suggest_int('n_estimators',500,2000),
#         "max_depth":trial.suggest_int('max_depth',5,50),
#         "learning_rate" : trial.suggest_float('learning_rate',1e-4, 0.1, log=True),
#         "min_child_weight" : trial.suggest_float('min_child_weight', 0.5,4),
#         "min_child_samples" : trial.suggest_int('min_child_samples',1,250),
#         "subsample" : trial.suggest_float('subsample', 0.2, 1),
#         "subsample_freq" : trial.suggest_int('subsample_freq',0,5),
#         "colsample_bytree" : trial.suggest_float('colsample_bytree',0.2,1),
#         'num_leaves' : trial.suggest_int('num_leaves', 8, 64),
#     }
#     lgbmopt = LGBMClassifier(**params,random_state=seed,device="gpu")
#     cv = cross_val_score(lgbmopt, X, y, cv = 4,scoring='roc_auc').mean()
#     return cv

# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=100,timeout=2000)

In [ ]:
lgbm_params = {'max_depth': 41, 'learning_rate': 0.03432850637422446,
               'min_child_weight': 2.9603503357916763, 'min_child_samples': 30,
               'subsample': 0.8782988886358021, 'subsample_freq': 3,
               'colsample_bytree': 0.501275718332705, 'num_leaves': 25}

lgbmmodel = LGBMClassifier(**lgbm_params,n_estimators = 1464,random_state=seed,device="gpu",verbose=-1)
lgbmmodel.fit(X_train,y_train)
print("CV score of LGBM is ",cross_val_score(lgbmmodel,X,y,cv=4, scoring = 'roc_auc').mean())
print("ROC AUC over Val Data:",roc_auc_score(lgbmmodel.predict(X_val),y_val))

<div id="4.4" >
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #263A29; font-weight: bold; font-size: 36px;">
   4.4 Optuna-Tuned XGB Classifier
    </h1>
</div>
<hr>

In [ ]:
# def objective(trial):
#     params = {
#     'n_estimators' : trial.suggest_int('n_estimators',1500,2500),
#     'max_depth':  trial.suggest_int('max_depth',3,8),
#     'min_child_weight': trial.suggest_float('min_child_weight', 2,4),
#     "learning_rate" : trial.suggest_float('learning_rate',1e-4, 0.2),
#     'subsample': trial.suggest_float('subsample', 0.2, 1),
#     'gamma': trial.suggest_float("gamma", 1e-4, 1.0),
#     "colsample_bytree" : trial.suggest_float('colsample_bytree',0.2,1),
#     "colsample_bylevel" : trial.suggest_float('colsample_bylevel',0.2,1),
#     "colsample_bynode" : trial.suggest_float('colsample_bynode',0.2,1),
#     }
    
#     xgbopt = XGBClassifier(**params,random_state=seed,tree_method = "gpu_hist",eval_metric= "auc")
#     cv = cross_val_score(xgbopt, X, y, cv = 4,scoring='roc_auc').mean()
#     return cv

# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=100,timeout=5000)

In [ ]:
xgb_params = {'max_depth': 6, 'min_child_weight': 2.7526603493948096,
              'learning_rate': 0.015273686530995825, 'subsample': 0.7109136660293711,
              'gamma': 0.2939927860245192, 'colsample_bytree': 0.5015735880252528,
              'colsample_bylevel': 0.6877145513802184, 'colsample_bynode': 0.9449955113410351}

xgbmodel = XGBClassifier(**xgb_params,n_estimators=1670, random_state=seed,tree_method= 'gpu_hist')
xgbmodel.fit(X_train,y_train)
print("CV score of XGB is ",cross_val_score(xgbmodel,X,y,cv=4, scoring = 'roc_auc').mean())
print("ROC AUC over Val Data:",roc_auc_score(xgbmodel.predict(X_val),y_val))

<div id="4.5" >
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #263A29; font-weight: bold; font-size: 36px;">
   4.5 Out-of-Fold Predictions LGBM + CatBoost + XGB
    </h1>
</div>
<hr>

In [ ]:
submission = pd.DataFrame()
submission["id"] = test_data.index
submission["Exited"] = 0

In [ ]:
SPLITS = 4
REPEATS = 2
lgbm_auc_score = []
cat_auc_score = []
xgb_auc_score = []
denom = 0

In [ ]:
for i,(tr,val) in enumerate(RepeatedStratifiedKFold(n_splits=SPLITS, n_repeats=REPEATS,random_state=seed).split(X,y)):
    
    print("-"*30,f"FOLD {i+1}/{SPLITS*REPEATS}","-"*30)
    X_train, X_test, y_train, y_test = X.iloc[tr,:],X.iloc[val,:],y.iloc[tr],y.iloc[val]
    
    print("\n->","LGBM:")
    lgbmmodel = LGBMClassifier(**lgbm_params,n_estimators= 7500,random_state=seed,device="gpu")
    lgbmmodel.fit(X_train,y_train, eval_set=[(X_test,y_test)], eval_names=["valid"],eval_metric=['auc'], early_stopping_rounds=2000,verbose = 1000)
    auc = roc_auc_score(y_test, lgbmmodel.predict_proba(X_test)[:,1])
    lgbm_auc_score.append(auc)
    print(f"\nFold {i+1} ROC_AUC of LGBM =", auc,"\n")
    submission["Exited"] += lgbmmodel.predict_proba(test_data)[:,1]
    denom+=1
    
    print("\n->","CAT:")
    train_dataset = Pool(data=X.iloc[tr,:],label=y.iloc[tr])
    eval_dataset = Pool(data=X.iloc[val,:],label=y.iloc[val])
    
    catmodel = CatBoostClassifier(iterations=7500,verbose=1000, od_type="Iter",eval_metric="AUC", random_seed=seed,early_stopping_rounds=2000)
    catmodel.fit(train_dataset, use_best_model=True, eval_set=eval_dataset)
    auc = roc_auc_score(y.iloc[val], catmodel.predict_proba(X.iloc[val,:])[:,1])
    cat_auc_score.append(auc)
    submission["Exited"] += catmodel.predict_proba(test_data)[:,1]
    denom+=1
    
    print("\n->","XGB:")
    xgbmodel = XGBClassifier(**xgb_params,n_estimators= 7500,random_state=seed, tree_method= 'gpu_hist',eval_metric="auc",early_stopping_rounds = 2000)
    xgbmodel.fit(X_train,y_train, eval_set=[(X_test,y_test)],verbose = 1000,callbacks=[EarlyStopping(rounds = 2000,save_best=True)])
    auc = roc_auc_score(y_test, xgbmodel.predict_proba(X_test)[:,1])
    xgb_auc_score.append(auc)
    print(f"\nFold {i+1} ROC_AUC of XGB =", auc)
    submission["Exited"] += xgbmodel.predict_proba(test_data)[:,1]
    denom+=1
    
print("\n\n","-"*50,sep="")
print("CV score of LGBM is ",np.array(lgbm_auc_score).mean())
print("CV score of CAT is ",np.array(cat_auc_score).mean())
print("CV score of XGB is ",np.array(xgb_auc_score).mean())

<div id="7" style="background-color: #e1d9ce; padding: 20px; border-radius: 20px; border: 2px solid black;">
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #000000; font-weight: bold; font-size: 42px;">
   Creating 'submission.csv'
    </h1>
</div>

In [ ]:
submission["Exited"] = submission["Exited"]/denom

In [ ]:
submission["Exited"] = 0.6*submission["Exited"]+0.2*public_work1["Exited"]+0.2*public_work2["Exited"]

submission.to_csv("submission.csv",header=True,index=False)
submission